# Dozer (machine 5): track final-drive gear wear — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | CORA vibration (Vx, 3 kHz) + T1 temperature | CORA infrared camera |
| Classes | healthy, wear 25%, wear 50%, wear 75% | same four |
| Tested on | 16 runs, one speed held out at a time | the same 16 runs |
| Code | `Models/dozer.py csv` | `Models/dozer.py cnn-pretrained` |

**Important:** there is no public dozer dataset with matching sensor and camera data. As planned in the dataset manifest, the Dozer final drive uses the **same 16 CORA gear-wear runs as the Belt Conveyor (not new data)** with the **same 4 wear classes**, so its scores are expected to **equal the Belt Conveyor's**: it maps that rig to the dozer track final drive and is not separate evidence. Lab data, not a dozer; it classifies the present condition, not when to repair.

All of this machine's experiment code is in `Models/dozer.py` (data loading shared with `Models/belt_conveyor.py`); shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [ ]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: CORA T1 samples", "Processed_Data/cora_gear_wear/t1_bounded_samples.npz", "*", 1),
    ("Sensor: CORA vibration (Vx)", "Raw_Data/cora_vibration", "*_Vx.csv", 16),
    ("Camera: CORA infrared stacks", "Raw_Data/cora_gear_wear", "*.mat", 16),
])

## 1. Sensor: vibration and temperature (CSV)
Each run's vibration becomes 1-second windows measured in **shaft orders** (multiples of the shaft speed), optionally with the run's T1 temperature. Gear-tooth wear shows at higher orders and in spikes. The features go into a CSV table; the model and feature set are chosen on validation speeds inside the training data, then tested once on the held-out speed.

In [ ]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "csv", tail=2)                    # main: unseen speed
    run_script("Models/dozer.py", "csv", "--split", "time", tail=1)  # secondary: time split
csv_result, time_result = load("dozer_csv_4class.json"), load("dozer_csv_4class_timesplit.json")
show({"MAIN (unseen speed) | Sensor: validation-selected": csv_result["validation_selected"]["score"],
      **{f"MAIN (unseen speed) | Sensor: {k}": v for k, v in csv_result["models"].items()},
      **{f"TIME SPLIT | Sensor: {k}": v for k, v in time_result["models"].items()}})

![CSV sensor model](../Figures/dozer/dozer_csv_4class.png) ![Time split](../Figures/dozer/dozer_csv_4class_timesplit.png)

**Two tests, clearly separated.** *Main (unseen speed):* each whole speed is held out (4 runs, one per wear level), so the model must recognise the wear level at a speed it never trained on. *Time split (secondary, easier):* the first 60% of every 50-minute run trains, the last 30% tests (10% gap); the same recordings appear in training, so part of a high score may come from recognising the recording rather than the wear.

## 2. Camera: infrared images with a pretrained CNN
`export-images` saves 150 infrared frames per run as `Images/dozer/<class>/dozer_<class>_<speed>_fNNN.png`. A pretrained ResNet18 is fine-tuned with data augmentation; the number of training rounds is chosen on validation runs. Retraining takes about half an hour, so it is off by default.

In [ ]:
TRAIN_CNN = False
if TRAIN_CNN and all(LOCAL.values()):
    run_script("Models/dozer.py", "export-images", tail=1)
    run_script("Models/dozer.py", "cnn-pretrained", tail=1)
    run_script("Models/dozer.py", "cnn-pretrained", "--split", "time", tail=1)
rows = {}
for name, label in (("dozer_cnn_pretrained_4class.json", "MAIN (unseen speed) | Camera: pretrained ResNet18"),
                    ("dozer_cnn_pretrained_4class_timesplit.json", "TIME SPLIT | Camera: pretrained ResNet18")):
    r = load(name)
    if r:
        rows[f"{label} (per run)"] = r["recording_level"]
show(rows)

![Pretrained CNN](../Figures/dozer/dozer_cnn_pretrained_4class.png) ![Time split](../Figures/dozer/dozer_cnn_pretrained_4class_timesplit.png)

**Data (CC BY 4.0):** CORA v2.1, DOI 10.34810/DATA2500, the same files as the Belt Conveyor.

## Proposal Week 6: advanced ensemble models (XGBoost / LightGBM)
The same CSV sensor features and the same held-out recordings, now with **XGBoost** and **LightGBM** next to random forest, extra trees and scikit-learn gradient boosting. Settings are fixed in advance; nothing is tuned on the test. `train_seconds` is the total training time.

In [ ]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "ensembles", tail=1)
    run_script("Models/dozer.py", "ensembles", "--split", "time", tail=1)
show_ensembles("dozer")

![Ensemble comparison](../Figures/dozer/dozer_ensembles.png)

## Week 7: anomaly detection (trained on healthy data only)
Isolation Forest and One-Class SVM learn only what **healthy** looks like; anything unusual raises an alarm. No fault labels are used for training. The alarm threshold is set from the healthy training recordings before testing. **ROC-AUC** (1.0 = perfect, 0.5 = guessing) shows how well the score separates healthy from faulty; **fault detection rate** and **false alarm rate** are at that fixed threshold.

In [ ]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "anomaly", tail=1)
    run_script("Models/dozer.py", "anomaly", "--split", "time", tail=1)
show_anomaly("dozer")

![Anomaly scores](../Figures/dozer/dozer_anomaly.png)

Each dot is one test recording; above the dashed line = alarm. Green = healthy, red = faulty.

## Try your own CSV
Put the path of any CSV in `CSV_FILE` and run this cell. It can be a **feature table** (same columns as `Processed_Data/dozer_sensor_features_4class.csv`, optional `class` column) or a **raw recording** (one column of CORA Vx vibration values). Set `LABEL` to the true class to see the accuracy. Classes: healthy, wear25, wear50, wear75. The saved model is trained on all project data, so testing on one of the project's own recordings gives an optimistic score; to test fairly, first run `python Models/dozer.py train-final --exclude <recording>`.

In [ ]:
CSV_FILE = None   # e.g. r"C:/Users/me/Downloads/my_recording.csv"
LABEL = None      # true class if known
SPEED = None      # raw recordings only: "F5", "F15", "F50" or "F60"
if CSV_FILE:
    run_script("Models/dozer.py", "predict", "--csv", CSV_FILE, *((["--label", LABEL] if LABEL else []) + (["--speed", SPEED] if SPEED else [])), tail=25)
else:
    print("Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).")